# Approach 1 — Rules-Based Archetype Classification

## What this approach does

Rules-based classification assigns each practice to a size band and a model band using
**explicit, hard-coded thresholds** derived from the framework criteria and the observed
data distribution. There is no training step — the logic is deterministic and produces
the same label every time for the same input.

## Why use it?

| Strength | Detail |
|----------|--------|
| Transparent | Every label can be fully explained to a non-technical audience |
| Fast to run | No fitting step — runs in milliseconds on any size dataset |
| Easy to tune | Thresholds are plain numbers you can update without rebuilding a model |
| Business-owned | The framework owners can adjust the rules to reflect policy decisions |
| Stable | Labels don't change unless you deliberately change the thresholds |

## When it works best

Rules-based classification is the right default when the business already has a clear
conceptual definition of each archetype and you want the labels to *enforce* that
definition rather than discover something different from the data.

It is less suitable when the boundaries are genuinely ambiguous, when the data
distribution changes materially over time, or when you want to discover structure
that your framework did not anticipate.

## N/A zones

The framework explicitly forbids certain Size × Model combinations:
- **NHS Led** cannot be **Large/Advanced** or **Flagship** — a practice at that scale
  cannot be viable on NHS activity alone.

Practices that fall into these cells are automatically reclassified to **Balanced Mixed**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

BLUE   = '#4C72B0'
ORANGE = '#DD8452'
GREEN  = '#55A868'
RED    = '#C44E52'
GREY   = '#8C8C8C'

sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

NHS_VALUE_PER_UDA = 28.0
SIZE_ORDER  = ['Small/Foundation', 'Medium/Core', 'Large/Advanced', 'Flagship']
MODEL_ORDER = ['NHS Led', 'Balanced Mixed', 'Private Led Mixed', 'Specialist/Referral Hub']
NA_ZONE_PAIRS = {('NHS Led', 'Large/Advanced'), ('NHS Led', 'Flagship')}

df = pd.read_csv('master.csv')
print(f'Loaded {len(df):,} practices')

---
## Feature engineering

Before classifying, we derive the signals the rules will use.

**NHS income proxy** — The dataset does not always have a direct NHS revenue figure.
UDA (Units of Dental Activity) contracted volume multiplied by the standard UDA rate
(£28) is the most reliable proxy available. When `nooftreatmentitems_nhs_standard` is
populated in your real data you can replace this proxy with the direct figure.

**NHS share** — The ratio `nhs_income / (nhs_income + private_income)` captures the
practice's position on the NHS–private spectrum as a single 0–1 value.

**Specialist proxy** — Without real `countof_snareid` or `chargeprice_private_referral`
data the best available signal is **hygienist presence** combined with **high private
income per chair**. Both indicate a practice invested in elective/specialist work.

In [ ]:
df['nhs_income_est']   = df['uda'].fillna(0) * NHS_VALUE_PER_UDA
df['private_income']   = df['privateincome'].fillna(0)
df['total_income_est'] = df['private_income'] + df['nhs_income_est']

df['nhs_share'] = np.where(
    df['total_income_est'] > 0,
    df['nhs_income_est'] / df['total_income_est'],
    np.nan
)

df['has_hygienist']            = df['position_hygienist'] > 0
df['private_income_per_chair'] = df['private_income'] / df['numberofchairs'].replace(0, np.nan)
hi_private_thresh              = df['private_income_per_chair'].quantile(0.75)
df['high_private_intensity']   = df['private_income_per_chair'] >= hi_private_thresh
df['specialist_flag']          = df['has_hygienist'] & df['high_private_intensity']

# Quick summary of the signals
print('NHS share distribution:')
print(df['nhs_share'].describe().round(3))
print(f'\nPractices with specialist flag: {df["specialist_flag"].sum()} ({df["specialist_flag"].mean()*100:.1f}%)')

---
## Size classification

### Logic

Size is primarily determined by `numberofsurgeries` — the number of treatment rooms.
This is the most stable and directly observable measure of physical capacity.

A secondary override promotes a practice to **Flagship** if it has six or more
surgeries *and* a high staff headcount (≥15), capturing large multi-dentist sites that
punch above the surgery-count threshold in workforce terms.

```
Surgeries   Band
---------   ----
  1 – 3     Small / Foundation
  4 – 5     Medium / Core
  6 – 7     Large / Advanced
  8 +        Flagship
  ≥6 AND staff ≥15  → Flagship (override)
```

**Calibration tip:** Run the distribution cell below and compare against your portfolio.
If your portfolio skews larger, shift the cut-offs right. The intent of the framework
is that each band represents roughly a quartile of the portfolio.

In [ ]:
def classify_size(df):
    s  = df['numberofsurgeries']
    st = df['unique_staff_ids']
    size = pd.Series('Medium/Core', index=df.index)
    size[s <= 3]                 = 'Small/Foundation'
    size[(s >= 4) & (s <= 5)]    = 'Medium/Core'
    size[(s >= 6) & (s <= 7)]    = 'Large/Advanced'
    size[s >= 8]                 = 'Flagship'
    size[(s >= 6) & (st >= 15)]  = 'Flagship'   # staff override
    return size

df['archetype_size'] = classify_size(df)

# Distribution check
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Size Classification — Distribution Check', fontweight='bold')

axes[0].hist(df['numberofsurgeries'], bins=range(1, 12), color=BLUE, edgecolor='white', align='left')
for cut, label in [(3.5, '3|4'), (5.5, '5|6'), (7.5, '7|8')]:
    axes[0].axvline(cut, color=RED, lw=1.5, ls='--')
    axes[0].text(cut + 0.1, axes[0].get_ylim()[1] * 0.9, label, color=RED, fontsize=8)
axes[0].set_xlabel('Number of surgeries')
axes[0].set_ylabel('Practices')
axes[0].set_title('Surgery distribution with cut-offs')

counts = df['archetype_size'].value_counts().reindex(SIZE_ORDER)
colors = sns.color_palette('Set2', 4)
bars = axes[1].bar(counts.index, counts.values, color=colors, edgecolor='white')
axes[1].bar_label(bars, padding=3)
axes[1].set_title('Practices per size band')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

print(df['archetype_size'].value_counts().reindex(SIZE_ORDER).to_string())

---
## Model classification

### Logic

Model classification captures *how* a practice earns its income — its commercial orientation.

The primary signal is **NHS share of income**. Rather than hard-coding fixed percentage
thresholds (which would produce very skewed segments if the portfolio-wide average shifts),
the bands are anchored to **percentiles of the portfolio distribution**. This ensures the
four model bands always represent meaningfully different parts of the income spectrum
regardless of whether you are looking at a predominantly NHS or predominantly private portfolio.

```
NHS share          Band
---------          ----
  >= P75           NHS Led
  P50 – P75        Balanced Mixed
  P25 – P50        Private Led Mixed
  < P25 + signals  Specialist / Referral Hub
  < P10            Specialist / Referral Hub (extreme private)
```

**Specialist / Referral Hub** requires either:
- NHS share below the portfolio P25 **and** the specialist proxy fires (hygienist present
  + top-quartile private income per chair), or
- NHS share below P10 regardless of specialist flag (extreme private concentration)

When `countof_snareid` or `nooftreatmentitems_private_referral` are populated in your
real data, replace or supplement the `specialist_flag` logic with those direct signals.

In [ ]:
def classify_model(df):
    q10 = df['nhs_share'].quantile(0.10)
    q25 = df['nhs_share'].quantile(0.25)
    q50 = df['nhs_share'].quantile(0.50)
    q75 = df['nhs_share'].quantile(0.75)

    print(f'NHS share percentiles used as thresholds:')
    print(f'  P10 = {q10:.3f}  |  P25 = {q25:.3f}  |  P50 = {q50:.3f}  |  P75 = {q75:.3f}')

    model = pd.Series('Balanced Mixed', index=df.index)
    model[df['nhs_share'] >= q75]                                = 'NHS Led'
    model[(df['nhs_share'] >= q50) & (df['nhs_share'] < q75)]   = 'Balanced Mixed'
    model[(df['nhs_share'] >= q25) & (df['nhs_share'] < q50)]   = 'Private Led Mixed'
    specialist_mask = (
        ((df['nhs_share'] < q25) & df['specialist_flag']) |
        (df['nhs_share'] < q10)
    )
    model[specialist_mask] = 'Specialist/Referral Hub'
    return model

df['archetype_model_raw'] = classify_model(df)

# N/A zone enforcement
df['archetype_model'] = df['archetype_model_raw'].copy()
for model_val, size_val in NA_ZONE_PAIRS:
    mask = (df['archetype_model_raw'] == model_val) & (df['archetype_size'] == size_val)
    reclassified = mask.sum()
    df.loc[mask, 'archetype_model'] = 'Balanced Mixed'
    if reclassified:
        print(f'  N/A zone: reclassified {reclassified} [{model_val} x {size_val}] -> Balanced Mixed')

df['archetype_rules'] = df['archetype_size'] + ' | ' + df['archetype_model']

# Distribution check
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Model Classification — Distribution Check', fontweight='bold')

axes[0].hist(df['nhs_share'].dropna(), bins=30, color=ORANGE, edgecolor='white')
for pct in [df['nhs_share'].quantile(q) for q in [0.1, 0.25, 0.5, 0.75]]:
    axes[0].axvline(pct, color=RED, lw=1.2, ls='--')
axes[0].set_xlabel('NHS share of income')
axes[0].set_ylabel('Practices')
axes[0].set_title('NHS share distribution with percentile cut-offs')

counts = df['archetype_model'].value_counts().reindex(MODEL_ORDER)
colors = sns.color_palette('Set2', 4)
bars = axes[1].bar(counts.index, counts.values, color=colors, edgecolor='white')
axes[1].bar_label(bars, padding=3)
axes[1].set_title('Practices per model band')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

---
## 4 × 4 Matrix — the full archetype picture

The 16-cell heatmap below shows the count of practices in every Size × Model combination.
Grey cells with **N/A** are invalid combinations per the framework and will always be zero
after the reclassification step.

In [ ]:
ct = pd.crosstab(df['archetype_size'], df['archetype_model']).reindex(
    index=SIZE_ORDER, columns=MODEL_ORDER, fill_value=0
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Rules-Based Archetype Matrix', fontsize=13, fontweight='bold')

# Count heatmap
sns.heatmap(ct, ax=axes[0], annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.7})
axes[0].set_title('Practice count')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Size')
axes[0].tick_params(axis='x', rotation=20)
axes[0].tick_params(axis='y', rotation=0)
for model_val, size_val in NA_ZONE_PAIRS:
    ri, ci = SIZE_ORDER.index(size_val), MODEL_ORDER.index(model_val)
    axes[0].add_patch(plt.Rectangle((ci, ri), 1, 1, fill=True, color='#d0d0d0', zorder=3, alpha=0.8))
    axes[0].text(ci + 0.5, ri + 0.5, 'N/A', ha='center', va='center', fontsize=9, color='grey', zorder=4)

# Average total income heatmap
income_pivot = df.groupby(['archetype_size', 'archetype_model'])['total_income_est'].mean().unstack(fill_value=0)
income_pivot = income_pivot.reindex(index=SIZE_ORDER, columns=MODEL_ORDER, fill_value=0)
annot = income_pivot.applymap(lambda v: f'£{v/1e3:.0f}k' if v > 0 else '')
sns.heatmap(income_pivot, ax=axes[1], annot=annot, fmt='', cmap='YlOrRd',
            linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.7})
axes[1].set_title('Average total income')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Size')
axes[1].tick_params(axis='x', rotation=20)
axes[1].tick_params(axis='y', rotation=0)
for model_val, size_val in NA_ZONE_PAIRS:
    ri, ci = SIZE_ORDER.index(size_val), MODEL_ORDER.index(model_val)
    axes[1].add_patch(plt.Rectangle((ci, ri), 1, 1, fill=True, color='#d0d0d0', zorder=3, alpha=0.8))
    axes[1].text(ci + 0.5, ri + 0.5, 'N/A', ha='center', va='center', fontsize=9, color='grey', zorder=4)

plt.tight_layout()
plt.show()

print('\nCrosstab (counts):')
print(ct.to_string())

---
## Archetype profiles

For each populated cell in the matrix, the table below shows the mean values of the
key operational metrics. This is the primary output for business review — it tells you
what a practice in each archetype *typically looks like* and where the biggest income
and productivity differences lie between segments.

In [ ]:
df['items_per_surgery'] = df['nooftreatmentitems'] / df['numberofsurgeries'].replace(0, np.nan)

profile = (
    df.groupby(['archetype_size', 'archetype_model'], observed=True)
    .agg(
        n=('practicekey', 'count'),
        avg_surgeries=('numberofsurgeries', 'mean'),
        avg_staff=('unique_staff_ids', 'mean'),
        avg_private_income=('private_income', 'mean'),
        avg_nhs_income=('nhs_income_est', 'mean'),
        avg_total_income=('total_income_est', 'mean'),
        avg_nhs_share_pct=('nhs_share', lambda x: x.mean() * 100),
        avg_items_per_surgery=('items_per_surgery', 'mean'),
        avg_uda=('uda', 'mean'),
    )
    .round(1)
)
display(profile)

# Income profile chart
model_colors = dict(zip(MODEL_ORDER, sns.color_palette('Set2', 4)))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Archetype Profiles — Income by Size Band', fontsize=13, fontweight='bold')

for model_val in MODEL_ORDER:
    sub = profile.reset_index()
    sub = sub[sub['archetype_model'] == model_val].set_index('archetype_size').reindex(SIZE_ORDER)
    axes[0].plot(SIZE_ORDER, sub['avg_total_income'], marker='o', lw=2,
                 color=model_colors[model_val], label=model_val)
    axes[1].plot(SIZE_ORDER, sub['avg_nhs_share_pct'], marker='o', lw=2,
                 color=model_colors[model_val], label=model_val)

axes[0].set_ylabel('Average total income (£)')
axes[0].set_title('Total income by size')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k'))
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', rotation=15)

axes[1].set_ylabel('Average NHS share (%)')
axes[1].set_title('NHS income share by size')
axes[1].legend(fontsize=8)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

---
## Save output

In [ ]:
out_cols = [
    'practicekey', 'practicename', 'region',
    'numberofsurgeries', 'unique_staff_ids',
    'private_income', 'nhs_income_est', 'total_income_est', 'nhs_share',
    'archetype_size', 'archetype_model', 'archetype_rules',
]
df[out_cols].to_csv('archetypes_rules.csv', index=False)
print(f'Saved archetypes_rules.csv  ({len(df):,} rows)')
df[out_cols].head()